# L1c: Working with Floating-Point Types

Almost every number in this course is stored in a fixed number of bits. This lecture takes that storage apart: first for integers written in an arbitrary base, then for a `Float64`, whose sign, exponent, and fraction we will extract and reassemble by hand.

> __Learning Objectives:__
>
> By the end of this lecture, you should be able to:
> * __Recover a value from its digits:__ Compute the positional sum of a digit sequence in any base and recover the number that produced it, using the same rule for binary, octal, and decimal.
> * __Locate the fields of a `Float64`:__ Identify the sign, biased exponent, and stored fraction inside a 64-bit pattern, and explain what the implicit leading digit contributes.
> * __Respect the limits of a finite representation:__ Explain why a fixed number of bits forces approximation, and why that makes exact equality the wrong default when comparing computed results.

Let's get started!
___

## Setup, Data, and Prerequisites
First, we set up the computational environment by including the `Include.jl` file and loading any needed resources.

> The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, etc. For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/).

Let's set up our code environment:

In [ ]:
include(joinpath(@__DIR__, "Include.jl")); # include the Include.jl file

The course environment also loads [the `VLDataScienceMachineLearningPackage.jl` package](https://github.com/varnerlab/VLDataScienceMachineLearningPackage.jl); see [the documentation](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/). This notebook does not need it; this notebook uses only Julia's `Base` library. We start using the package later in the course.

___

## Theory: Base b representation of numbers
A number in base $b$ is represented by a finite sequence of digits $(d_{n}d_{n-1}\dots{d_{1}}d_{0})_{b}$ where each digit $d_{i}$ satisfies $0\leq d_{i} < b$. The value (in base 10) of a base-$b$ number is the positional sum:
$$
\begin{align*}
\underbrace{(d_{n}d_{n-1}\dots{d_{1}}d_{0})_{b}}_{\text{base b}} = \underbrace{\sum_{i=0}^{n}d_{i}b^{i}}_{\text{value in base 10}}
\end{align*}
$$
Let's use integers for a few examples to better understand this expression (and then we'll move on to floating-point numbers).

Consider an `Int64` number. We know that memory storage on modern (non-quantum) hardware is binary, i.e., base $b = 2$; thus, all the digits $d_{i}$ must satisfy $0\leq d_{i} < 2$.

However, how many digits do we have? Careful here: the sequence $(d_{n}d_{n-1}\dots d_{1}d_{0})$ holds $n+1$ digits because it starts counting at zero. So for an `Int64` the highest index is $n = 63$, and the `64` in `Int64` is the _word size_, one more than the top index.

> __Hmmm__. Didn't we already see that? Yes, it's the length of the string output from [the `bitstring(...)` function](https://docs.julialang.org/en/v1/base/numbers/#Base.bitstring)! Let's count the number of zero digits and the number of one digits of a test 64-bit integer using the [`count_zeros(...)` function](https://docs.julialang.org/en/v1/base/numbers/#Base.count_zeros) and the [`count_ones(...)` function](https://docs.julialang.org/en/v1/base/numbers/#Base.count_ones).

To check the equality condition, we use the [Julia @assert macro](https://docs.julialang.org/en/v1/base/base/#Base.@assert). If the statement passed to the [@assert macro](https://docs.julialang.org/en/v1/base/base/#Base.@assert) evaluates to `false`, i.e., the number of zeros and ones does not equal the `wordsize`, then an [AssertionError](https://docs.julialang.org/en/v1/base/base/#Core.AssertionError) is thrown, alerting us that there is an issue.

> __Note__: we use [the equality `==` operator](https://docs.julialang.org/en/v1/manual/missing/#Equality-and-Comparison-Operators) (not the assignment operator `=`). There is also [the `===` comparison operator](https://docs.julialang.org/en/v1/manual/missing/#Equality-and-Comparison-Operators) in Julia, which determines whether `x` and `y` are identical in the sense that no program could distinguish them. We'll see this operator later.

So, what do we see?

In [ ]:
let
    wordsize = 64; # default word size
    x = 18; # pick an integer value (Int64 value by default)
    n = count_zeros(x) + count_ones(x); # this counts 0's and 1's (doesn't give any info about position)
    @assert wordsize == n # see https://docs.julialang.org/en/v1/base/base/#Base.@assert
end

### Binary numbers
We can get the bit pattern (binary representation) of an integer by calling [the `bitstring(...)` function](https://docs.julialang.org/en/v1/base/numbers/#Base.bitstring), but there is a __wrinkle__.

> __Wrinkle__: the [`bitstring(...)` function](https://docs.julialang.org/en/v1/base/numbers/#Base.bitstring) returns the bit pattern as a `String`. We'll have to convert that `String` to an array of `0` and `1` to do any computation with these values. More on that shortly.

The positions of the `0` and `1` values in the binary number give the number's value. Suppose we get the bit pattern, i.e., the positions of the digits of some integer value `x::Int`, using [the `bitstring(...)` function](https://docs.julialang.org/en/v1/base/numbers/#Base.bitstring) and save this value in the `sₒ::String` variable.

In [ ]:
sₒ,xₒ = let
    x = 100456; # Int64 value by default
    s = bitstring(x)
    s,x
end

For the binary string $s$, we sum powers of 2 (the $b^{i}$ terms in the sum) for positions whose digit is `1`, processing the string from right to left. Let's make this more concrete.

> __Hypothesis__: We should be able to process the string `s` (compute its positional sum) and recover the integer that generated it. To do this we'll use a few techniques we haven't covered yet. Don't worry about the implementation for now.  

To check our hypothesis, we need to do a few things. The first is to convert the bit pattern in `s::String` into an array of numbers (so we can compute the positional sum). 

The following logic contains a few advanced things, e.g., working with arrays and [`String` and `Char` types](https://docs.julialang.org/en/v1/manual/strings/#man-strings), function piping (`|>`), etc.; don't worry too much about the details yet:

In [ ]:
bit_pattern_array = sₒ |> collect |> reverse .|> v-> parse(Int,v) # This is a magical line!

__Hmmm__. Okay, we can convert the string to an `Array{Int64,1}`, which is good. However, arrays in Julia are `1`-based, meaning the first index in the array occurs at index `1`. But our positional expressions assume zero-based indexing (first value at index `0`). Can we make a zero-based array in Julia?

* __Hack__: Yes, we can copy `bit_pattern_array` into a dictionary (which we can make 0-based), called `bit_pattern_dictionary::Dict{Int64,Int64}`. This allows us to start counting from 0 instead of 1.
* __Proper solution__: In addition to this hack (which is convenient), a cleaner solution is to use [an `OffsetArray` from the `OffsetArrays.jl` package](https://github.com/JuliaArrays/OffsetArrays.jl) to fix the 1-based indexing.

Let's use `0`-based dictionary hack to populate a dictionary with `bit_pattern_array` values (indexed from zero).

In [ ]:
bit_pattern_dictionary = let
    bit_pattern_dictionary = Dict{Int64,Int64}(); # Declare memory
    for i ∈ eachindex(bit_pattern_array)
        bit_pattern_dictionary[i-1] = bit_pattern_array[i] # what are we doing here?
    end
    bit_pattern_dictionary; # return the 0-based mapping
end

In [ ]:
let

    b = 2; # What base do we have?
    count = 0; # if this works, when we are finished, this should be our original number
    positions = keys(bit_pattern_dictionary) |> collect |> sort; # what is going on here? (we're iterating the 0-based bit_pattern_dictionary)
    for i ∈ positions
        dᵢ = bit_pattern_dictionary[i];
        count+= (dᵢ)*(b^i) # what is += doing?
    end
    println("Was your original number $(count)?")
end

### What about negative integers?

The positional sum we just wrote gives every digit a weight of $+b^{i}$. A sum of non-negative terms can never be negative, yet `L1a` told us `Int64` represents negative whole numbers too. So what happens if we hand the same algorithm a negative value?

> __Predict before you run:__ does `x = -18` come back as `-18`, as some enormous positive number, or as something else entirely?

Let's find out:

In [ ]:
let
    x = -18; # a negative Int64 this time

    # exactly the same construction as above -
    bpa = bitstring(x) |> collect |> reverse .|> v -> parse(Int, v)
    bpd = Dict{Int64,Int64}();
    for i ∈ eachindex(bpa)
        bpd[i-1] = bpa[i]
    end

    # exactly the same positional sum as above -
    b = 2; count = 0;
    for i ∈ (keys(bpd) |> collect |> sort)
        count += (bpd[i])*(b^i)
    end

    (recovered = count, original = x, agree = count == x)
end

It comes back as `-18`. But it is right for the __wrong reason__, and both halves of that are worth knowing:

__Signed integers use two's complement.__ There is no sign bit in the floating-point sense. In an `Int64` the top digit carries weight $-2^{63}$, not $+2^{63}$. The positional formula for a signed integer is given by:
$$
x = -d_{63}2^{63} + \sum_{i=0}^{62}d_{i}2^{i}
$$

__Integer overflow rescued our wrong formula.__ Our loop computed $+d_{63}2^{63}$. But `2^63` does not fit in an `Int64`; it wraps around to exactly $-2^{63}$, silently turning the wrong term into the right one. Julia does not warn you.

Swap the base for an arbitrary-precision one (`b = 2` becomes `b = big(2)`) and the overflow disappears along with the accident: the sum returns `18446744073709551598`, which is what the unsigned formula actually specifies. 

Getting the right answer for the wrong reason is worth seeing once, early, before you start trusting arithmetic that "obviously works."

#### Thought Question 1: Binary Representation
Why do you think computers use binary (base 2) instead of decimal (base 10) for representing numbers internally? What are the advantages and disadvantages of this choice? How might this affect the way we think about numerical computations in programming?

___

## Beyond binary numbers
There are many everyday applications for base $b>2$ numbers! Larger bases like decimal (base 10), dozenal (base 12), and sexagesimal (base 60) exist in everyday measurements and commerce. There are also a few others that you may encounter every day, but not realize it:
> __Hexadecimal (base 16)__ compactly encodes binary data for color codes; for example, Cornell red is `#B31B1B`, while base 32/64 are used to encode arbitrary binary data (e-mail attachments, URLs, certificates) into printable characters.

Though higher bases require a more complex digit set, they dramatically shorten the representation of large values.

#### Digits Example
Let's consider an octal (base 8) example. Instead of calling [the `bitstring(...)` function](https://docs.julialang.org/en/v1/base/numbers/#Base.bitstring) (which always returns a base $b=2$ value), let's explore [the `digits(...)` function](https://docs.julialang.org/en/v1/base/numbers/#Base.digits). The [`digits(...)` function](https://docs.julialang.org/en/v1/base/numbers/#Base.digits) takes a `number`, a `base`, and a `pad` argument, and returns the __digits of `number` written in `base`__, least significant digit first, zero-padded to at least `pad` digits.

> __`pad` is not a word size:__ it is a minimum digit count _in the base you asked for_. `pad = 16` gives at least 16 __octal__ digits, far more than any 16-bit value needs, since six octal digits already cover $8^{6} > 2^{16}$. We pass a generous `pad` only so the positional sum has a fixed range to loop over.

That leaves the question of what the digits themselves look like.

> __Octal__: Let's use [the `digits(...)` function](https://docs.julialang.org/en/v1/base/numbers/#Base.digits) to get the digits of $n = 74$ written in `base = 8`. Save this data in the `bit_pattern_array_octal::Vector{Int64}` variable.

So what do the octal digits look like?

In [ ]:
bit_pattern_array_octal = digits(74, base=8, pad=16) # octal digits of 74, least significant first

__Check__: Let's convert the octal number stored in the `bit_pattern_array_octal::Array{Int64,1}` variable back into base 10 by computing the positional sum in base 8.

In [ ]:
let
    # initialize -
    bit_pattern_dictionary = Dict{Int64,Int64}();
    b = 8.0; # base 8 for this example
    wordsize = 16;
    foreach(i -> bit_pattern_dictionary[i-1] = bit_pattern_array_octal[i], 
        eachindex(bit_pattern_array_octal)); # compact syntax for building bit dict

    # loop -
    value = 0.0;
    bitrangearray = range(0,stop=(wordsize-1),step=1) |> collect;
    for i ∈ bitrangearray
         dᵢ = bit_pattern_dictionary[i];
         value += (dᵢ)*(b^i)
    end

    value
end

#### Thought Question 2: Number Bases
We explored binary (base 2) and octal (base 8) representations. How does the choice of base affect the compactness of representing large numbers? Can you think of real-world applications where using a base other than 10 might be advantageous? What challenges might arise when converting between different bases?

___

## Floating-point numbers
Now that we have seen how integers are laid out in memory, let's explore floating-point formats: `Float16`, `Float32`, and `Float64`. In particular, we'll look at the memory layout of `Float64`.

Why have multiple floating-point precisions?
>Using multiple floating-point types lets us balance precision and resource usage for different applications:
> * `Float16` (half-precision) minimizes memory footprint at the expense of precision, which is useful for large-scale machine learning inference or graphics where fine precision isn't critical.
> * `Float32` (single-precision) offers a good compromise of speed and accuracy for many numerical and real-time workloads.
> * `Float64` (double-precision) provides high precision and a wide exponent range needed in scientific computing, simulations, and financial modeling where rounding errors must be controlled.

If we need more precision than `Float64`, Julia's `Base` already provides [the arbitrary-precision `BigFloat` type](https://docs.julialang.org/en/v1/base/numbers/#Base.MPFR.BigFloat-Tuple%7BAny,%20RoundingMode%7D), and packages such as `Quadmath.jl` add fixed-width types like `Float128`.

### Example: Memory Layout Float64

<div>
    <center>
        <img src="figs/Fig-64-bit-label-pattern.svg" width="580"/>
    </center>
</div>

Recall from `L1a` that a floating-point value carries three parts: a sign, an exponent (the scale), and a significand. Now we can see exactly where each one lives in memory.

Suppose we have a floating-point number $x\in\mathbb{R}$ that is approximated as a 64-bit value in memory. A __finite, normalized__ 64-bit value $x\in\mathbb{R}$ is encoded in memory as:
$$
\begin{align*}
x = \underbrace{S}_{\text{sign}}\times\underbrace{\text{significand}}_{1\,+\,\text{stored fraction}}\times\underbrace{{2^{E-1023}}}_{\text{scale}}
\end{align*}
$$
where:
$$
\begin{align*}
S &= (-1)^{d_{63}}\\
\text{significand} &= 1 + \sum_{i = 1}^{52}d_{52-i}2^{-i}\\
E &= \sum_{i=52}^{62}d_{i}2^{i - 52}
\end{align*}
$$
> __What does this mean?__ The leading `1 +` is the implicit bit from `L1a`; it is nowhere in the 64 stored bits. That assumption fails for zero, for very small (subnormal) values, and for `Inf` and `NaN`, each of which is signalled by a reserved exponent pattern. We work out exactly which patterns are reserved, and watch this formula break on them, in [the `Float32` example](./CHEME-5800-L1c-Example-Float32Representation-Fall-2026.ipynb).

The 64- and 32-bit formats differ in the number of bits allocated to the significand and exponent and in the position of the sign bit; otherwise they follow the same structural layout.

Let's compute the components of an example 64-bit floating-point value and see if we can reconstruct the original number. First, choose a test value for $x$:

In [ ]:
x = -65.78912; # example 64-bit floating point number (negative, so we can watch the sign bit)

Next, we'll use [the `bitstring(...)` function](https://docs.julialang.org/en/v1/base/numbers/#Base.bitstring) to get the 64-bit binary String, then we'll convert that into a 0-based bit pattern dictionary which we save in the `d::Dict{Int64,Int64}` variable:

In [ ]:
d = let

    # initialize -
    bitpattern_dictionary = Dict{Int64,Int64}();
    wordsize = 64; # how big is the word size?
    a = bitstring(x) |> reverse |> collect .|> v-> parse(Int64,v) # fancy. Nothing to see here, move along (for now anyway).
    
    # put stuff in the bit pattern dictionary
    for i ∈ 0:(wordsize-1)
        bitpattern_dictionary[i] = a[i+1];
    end
    bitpattern_dictionary # return the dictionary
end

#### Sign term
Now that we have the bit pattern dictionary `d::Dict{Int64, Int64}`, we can compute the components of the 64-bit floating point number. Let's start with the sign value `S::Int64`:

In [ ]:
S = let
    s = d[63]; # sign bit is at d63
    S = (-1)^s # if d63 = 1, we'll have a negative number, d63 = 0 gives us a positive number
end

#### Significand
Next, we'll compute the significand using the expression above. We'll also check our computed value using [the `significand(...)` function](https://docs.julialang.org/en/v1/base/numbers/#Base.Math.significand) to make sure we are correct. We'll store our calculated value in the `calculated_significand_value::Float64` variable:

In [ ]:
calculated_significand_value = let

    calculated_significand_value = 0.0;
    b = 2.0; # binary, base = 2
    number_of_fraction_bits = 52; # the stored fraction is d[51] down to d[0]
    significand_range_array = range(1,stop=number_of_fraction_bits,step=1) |> collect; # the weights 2^-1 ... 2^-52

    # loop: process each bit in the significand_range_array -
    for i ∈ significand_range_array
        calculated_significand_value += (b^(-i))*d[number_of_fraction_bits-i]
    end
    calculated_significand_value + 1 # don't forget to add 1
end

__Check__: Let's use [the `@assert` macro](https://docs.julialang.org/en/v1/base/base/#Base.@assert) to check our calculated significand value against the output of [the `significand(...)` function](https://docs.julialang.org/en/v1/base/numbers/#Base.Math.significand) using [the `==` comparison operator](https://docs.julialang.org/en/v1/manual/missing/#Equality-and-Comparison-Operators). 
> __What happens?__ If [the `==` comparison](https://docs.julialang.org/en/v1/manual/missing/#Equality-and-Comparison-Operators) comes back `false`, [an `AssertionError` is thrown](https://docs.julialang.org/en/v1/base/base/#Core.AssertionError) (and we know something is wrong with our calculation):

The one wrinkle is the sign.

> __Why `abs`?__ Our formula builds the significand as `1 +` a sum of non-negative terms, so it is always positive; the sign is carried separately in $S$. Julia's [`significand(...)` function](https://docs.julialang.org/en/v1/base/numbers/#Base.Math.significand) folds the sign back in, returning `-1.027955` for our negative `x`. So we compare magnitudes.

So what happens?

In [ ]:
@assert abs(significand(x)) == calculated_significand_value # compare built-in versus our calculated value

#### Exponent scale term: 
Lastly, let's compute the exponent value $E$, which gives us the scale of the number. We'll save this value in the `E::Float64` variable:

In [ ]:
E = let

    # initialize 
    calculated_exponent_value = 0.0;
    b = 2.0; # binary, base = 2
    lsb = 52; # least significant bit
    msb = 62; # most significant bit
    exponent_bit_range_array = range(lsb,stop=msb, step = 1) |> collect; # range of bits for E

    # loop: Let's process each of the bits in exponent_bit_range_array -
    for i ∈ exponent_bit_range_array
        calculated_exponent_value += d[i]*(b^(i-lsb))
    end
    calculated_exponent_value # return
end

#### Do we get the same number $x$?
If our implementation is correct, we should be able to reconstruct the original 64-bit value $x$ from its bit pattern.

> We'll use [the `@assert` macro](https://docs.julialang.org/en/v1/base/base/#Base.@assert) to compare our reconstructed value with the original `x`. If the comparison fails, [an `AssertionError`](https://docs.julialang.org/en/v1/base/base/#Core.AssertionError) will indicate an issue with the calculation.

Do the three fields rebuild the number we started with?

In [ ]:
let
    our_calculated_value = S*calculated_significand_value*2^(E - 1023);
    @assert our_calculated_value == x # same value for x?
end

In [ ]:
our_calculated_value = S*calculated_significand_value*2^(E - 1023)

#### Thought Question 3: Floating-Point Precision
When would you choose `Float32` instead of `Float64`, and how could that choice change the result of an engineering calculation?

___

## Lab

In Lab `L1d` you will turn this by-hand decomposition into a function. Everything we did above with `let` blocks and a bit-pattern dictionary becomes one callable interface that reports the three fields and rebuilds the value from them.

## Want some more?

If you want another worked layout, this companion example repeats the whole calculation for a 32-bit value, where the smaller word size makes the precision and range tradeoffs sharper:

> [▶ Layout of a `Float32`](./CHEME-5800-L1c-Example-Float32Representation-Fall-2026.ipynb). We analyze the layout of a `Float32` in memory, derive its machine epsilon, and work out which exponent patterns are reserved for zero, subnormals, infinity, and `NaN`.

___

## Tests
The test code checks some values in your notebook and gives you feedback on which items are correct or different. `Unhide` the test code (if you are curious) to see how we implemented the tests and what we are testing.

In [ ]:
let
    @testset verbose = true "CHEME 4/5800 L1c Test Suite" begin

        @testset "Integer bitstring properties" begin
            x = 32
            @test length(bitstring(x)) == 64
            @test count_zeros(x) + count_ones(x) == 64
        end

        @testset "Binary positional sum recovery" begin
            x = 45678
            s = bitstring(x)
            bit_pattern_array = bitstring(x) |> collect |> reverse .|> x-> parse(Int,x)
            bit_pattern_dictionary = Dict{Int64,Int64}()
            for i ∈ eachindex(bit_pattern_array)
                bit_pattern_dictionary[i-1] = bit_pattern_array[i]
            end
            
            count = 0
            b = 2
            positions = keys(bit_pattern_dictionary) |> collect |> sort
            for i ∈ positions
                dᵢ = bit_pattern_dictionary[i]
                count += (dᵢ)*(b^i)
            end
            @test count == x
        end

        @testset "Octal conversion" begin
            n = 74
            base = 8
            pad = 16
            bit_pattern_array_octal = digits(n, base=base, pad=pad)
            bit_pattern_dictionary = Dict{Int64,Int64}()
            foreach(i -> bit_pattern_dictionary[i-1] = bit_pattern_array_octal[i], 
                eachindex(bit_pattern_array_octal))
            
            value = 0.0
            b = 8.0
            wordsize = 16
            bitrangearray = range(0,stop=(wordsize-1),step=1) |> collect
            for i ∈ bitrangearray
                 dᵢ = bit_pattern_dictionary[i]
                 value += (dᵢ)*(b^i)
            end
            @test value == n
        end

        @testset "Floating-point reconstruction" begin
            for x ∈ (3.1415926535897, -65.78912) # positive and negative, so the sign bit is exercised
                d = let
                    bitpattern_dictionary = Dict{Int64,Int64}()
                    wordsize = 64
                    a = bitstring(x) |> reverse |> collect .|> v-> parse(Int64,v)
                    for i ∈ 0:(wordsize-1)
                        bitpattern_dictionary[i] = a[i+1]
                    end
                    bitpattern_dictionary
                end

                S = (-1)^d[63]
                calculated_significand_value = let
                    calculated_significand_value = 0.0
                    b = 2.0
                    number_of_fraction_bits = 52
                    significand_range_array = range(1,stop=number_of_fraction_bits,step=1) |> collect
                    for i ∈ significand_range_array
                        calculated_significand_value += (b^(-i))*d[number_of_fraction_bits-i]
                    end
                    calculated_significand_value + 1
                end

                E = let
                    calculated_exponent_value = 0.0
                    b = 2.0
                    lsb = 52
                    msb = 62
                    exponent_bit_range_array = range(lsb,stop=msb, step = 1) |> collect
                    for i ∈ exponent_bit_range_array
                        calculated_exponent_value += d[i]*(b^(i-lsb))
                    end
                    calculated_exponent_value
                end

                our_calculated_value = S*calculated_significand_value*2^(E - 1023)
                @test our_calculated_value == x
            end
        end
    end
end;

___

## Summary
Both integers and floating-point numbers are positional representations in a fixed number of bits, and reading those bits directly explains most of the surprises they produce.

> __Key Takeaways:__
>
> * **One positional rule, with one exception:** Binary, octal, and decimal values are all recovered by the same sum of digits times powers of the base, but signed integers break the pattern because the top digit carries a negative weight under two's complement.
> * **A `Float64` is three fields, not one number:** A sign bit, a biased exponent, and a stored fraction combine to give the value, and reconstructing a number from those fields shows exactly where its precision comes from.
> * **Finite representations force approximate comparison:** Because a value is stored in a fixed number of bits, equality between computed floating-point results is the exception, and comparisons must respect the spacing of the representation.

The gap between the number you wrote and the number the machine stored is not an error you can eliminate. It is a property of the format, and knowing its size is what lets you decide when it matters.
___